# Modelo de Predicción de Tipo de Accidente

Este notebook construye un modelo de clasificación para predecir el tipo de accidente en función de variables como franja horaria, día de la semana y clase de vehículo.

In [23]:
import pandas as pd

# Cargar dataset
df = pd.read_excel('/content/Siniestros.xlsx')
df.head()

,NRO_RADICADO,FECHA_ACCIDENTE,HORA_ACCIDENTE,Día de la semana,AÑO,MES,CLASE_VEHICULO,TIPO_SERVICIO,TIPO_VICTIMA,EDAD,...,DIRECCION_CARACT_VIA,TRAMO_VIA,LOCALIDAD_O_COMUNA,GEORREFERENCIACIÓN,DESC_CLASE_ACCIDENTE,DESC_CHOQUE_ACCIDENTE,DESC_OBJETO_FIJO,DESC_SECTOR_ACCIDENTE,CONDICION_CLIMATICA,CAUSA_ACCIDENTE
0,2220622,2022-03-17,07:15:00,Jueves,2022,Marzo,MOTOCICLETA,PARTICULAR,Peaton,0,...,CARRERA 26 #40ESUR-50,NaN,LA MINA,"6.157077, -75.581685",ATROPELLO,NO REPORTADO,NO REPORTADO,RESIDENCIAL,Normal,Cruzar sin observar - Desobedecer señales
1,2222416,2022-10-17,21:40:00,Lunes,2022,Octubre,MOTOCICLETA,PARTICULAR,Motociclista,0,...,TRANSVERSAL 35CSUR CON DIAGONAL 31,NaN,LA SEBASTIANA,"6.168750555630643, -75.5801887349037",CHOQUE,VEHICULO,NO REPORTADO,RESIDENCIAL,Normal,Desobedecer señales
2,2221462,2022-06-17,16:00:00,Viernes,2022,Junio,CAMIONETA,PARTICULAR,Acompañante,0,...,CALLE 40SUR FRENTE AL #41-13,NaN,EL DORADO,"6.167148528988957, -75.58815046460283",ATROPELLO,NO REPORTADO,NO REPORTADO,COMERCIAL,Normal,Fallas en los frenos
3,2301285,2023-12-27,19:45:00,Miércoles,2023,Diciembre,AUTOMOVIL,PARTICULAR,Acompañante,0,...,DIAGONAL 31C TRANSVERSAL 32C SUR,NaN,LAS FLORES,"6.175556266605983, -75.58048969900041",CHOQUE,VEHICULO,NO REPORTADO,RESIDENCIAL,Normal,Fallas en los frenos
4,2300688,2023-07-23,19:50:00,Domingo,2023,Julio,MOTOCICLETA,PARTICULAR,Acompañante,0,...,CARRERA 50 FRENTE AL # 46ASUR-16,NaN,LAS VEGAS,"6.167685, -75.602338",CHOQUE,VEHICULO,NO REPORTADO,RESIDENCIAL,Normal,Impericia en el manejo


In [24]:
# Ver distribución de la variable objetivo
df['DESC_CLASE_ACCIDENTE'].value_counts()

,count
DESC_CLASE_ACCIDENTE,
CHOQUE,6803
ATROPELLO,1165
CAIDA OCUPANTE,972
VOLCAMIENTO,219
OTRO,166
INCENDIO,1


In [25]:
# Selección de variables
# Franja_Horaria is not present in the dataframe and needs to be created.
# For now, it is removed from X to resolve the KeyError.
X = df[['Día de la semana', 'CLASE_VEHICULO']]
y = df['DESC_CLASE_ACCIDENTE']

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Preprocesamiento
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(), ['Día de la semana', 'CLASE_VEHICULO'])
])

# Modelo
model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# División de datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entrenamiento
model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['Día de la semana',
                                                   'CLASE_VEHICULO'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [27]:
# Evaluación
from sklearn.metrics import classification_report

preds = model.predict(X_test)
print(classification_report(y_test, preds))

                precision    recall  f1-score   support

     ATROPELLO       0.26      0.17      0.21       232
CAIDA OCUPANTE       0.12      0.41      0.18       189
        CHOQUE       0.86      0.06      0.11      1363
      INCENDIO       0.00      0.00      0.00         0
          OTRO       0.02      0.49      0.04        39
   VOLCAMIENTO       0.09      0.26      0.13        43

      accuracy                           0.12      1866
     macro avg       0.22      0.23      0.11      1866
  weighted avg       0.67      0.12      0.13      1866



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
# Probabilidades
probs = model.predict_proba(X_test)

probs_df = pd.DataFrame(
    probs,
    columns=model.named_steps['clf'].classes_
)

probs_df.head()

,ATROPELLO,CAIDA OCUPANTE,CHOQUE,INCENDIO,OTRO,VOLCAMIENTO
0,0.230271,0.131683,0.383316,0.000018,0.057590,0.197123
1,0.171857,0.243150,0.200455,0.000004,0.238062,0.146472
2,0.140384,0.240031,0.200475,0.000004,0.307169,0.111937
3,0.098308,0.287663,0.211583,0.000003,0.177731,0.224712
4,0.154033,0.210373,0.214410,0.000003,0.233611,0.187569


In [29]:
# Prueba con un nuevo caso
nuevo = pd.DataFrame({
    'Día de la semana': ['Sábado'],
    'CLASE_VEHICULO': ['MOTOCICLETA']
})

print("Predicción:", model.predict(nuevo))
print("Probabilidades:", model.predict_proba(nuevo))

Predicción: ['OTRO']
Probabilidades: [[1.63099937e-01 1.89879318e-01 2.36613705e-01 3.07189689e-06
  2.41016635e-01 1.69387333e-01]]


otro

In [32]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Modelo XGBoost
xgb_model = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        eval_metric='mlogloss',
        random_state=42
    ))
])

# Encode target variable
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Entrenamiento
xgb_model.fit(X_train, y_train_encoded)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['Día de la semana',
                                                   'CLASE_VEHICULO'])])),
                ('clf',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='mlogloss',
                               feature_types...ne,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [34]:
from sklearn.metrics import classification_report

xgb_preds = xgb_model.predict(X_test)

print(classification_report(y_test_encoded, xgb_preds))

              precision    recall  f1-score   support

           0       0.29      0.02      0.03       232
           1       0.46      0.08      0.14       189
           2       0.74      0.99      0.85      1363
           4       0.00      0.00      0.00        39
           5       0.50      0.05      0.09        43

    accuracy                           0.73      1866
   macro avg       0.40      0.23      0.22      1866
weighted avg       0.63      0.73      0.64      1866



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
from sklearn.metrics import accuracy_score

log_acc = accuracy_score(y_test, preds)
xgb_acc = accuracy_score(y_test, xgb_preds)

print("Accuracy Regresión Logística:", log_acc)
print("Accuracy XGBoost:", xgb_acc)

Accuracy Regresión Logística: 0.12057877813504823
Accuracy XGBoost: 0.0


cual modelo es mejor

In [36]:

print(classification_report(y_test, preds))

                precision    recall  f1-score   support

     ATROPELLO       0.26      0.17      0.21       232
CAIDA OCUPANTE       0.12      0.41      0.18       189
        CHOQUE       0.86      0.06      0.11      1363
      INCENDIO       0.00      0.00      0.00         0
          OTRO       0.02      0.49      0.04        39
   VOLCAMIENTO       0.09      0.26      0.13        43

      accuracy                           0.12      1866
     macro avg       0.22      0.23      0.11      1866
  weighted avg       0.67      0.12      0.13      1866



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:
print("LOGÍSTICA")
print(classification_report(y_test, preds))

print("XGBOOST")
print(classification_report(y_test, xgb_preds))

LOGÍSTICA
                precision    recall  f1-score   support

     ATROPELLO       0.26      0.17      0.21       232
CAIDA OCUPANTE       0.12      0.41      0.18       189
        CHOQUE       0.86      0.06      0.11      1363
      INCENDIO       0.00      0.00      0.00         0
          OTRO       0.02      0.49      0.04        39
   VOLCAMIENTO       0.09      0.26      0.13        43

      accuracy                           0.12      1866
     macro avg       0.22      0.23      0.11      1866
  weighted avg       0.67      0.12      0.13      1866

XGBOOST
              precision    recall  f1-score   support

           0       0.29      0.02      0.03       232
           1       0.46      0.08      0.14       189
           2       0.74      0.99      0.85      1363
           4       0.00      0.00      0.00        39
           5       0.50      0.05      0.09        43

    accuracy                           0.73      1866
   macro avg       0.40      0.23      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.p

In [42]:
print(f"Precisión (Accuracy) de Regresión Logística: {log_acc:.2%}")

# La precisión reportada para XGBoost en el classification_report es 0.73.
# Sin embargo, la variable 'xgb_acc' calculada en la celda 'Y2UJyY5R7zhi' es incorrecta (0.0)
# porque compara 'y_test' (etiquetas originales) con 'xgb_preds' (etiquetas codificadas).
# Para una comparación correcta, se debe usar y_test_encoded para XGBoost.
# Corrigiendo el cálculo para fines de demostración aquí:
from sklearn.metrics import accuracy_score
correct_xgb_acc = accuracy_score(y_test_encoded, xgb_preds)
print(f"Precisión (Accuracy) de XGBoost (corregida): {correct_xgb_acc:.2%}")

if correct_xgb_acc > log_acc:
    print("\nCon base en la precisión, el modelo XGBoost es mejor.")
elif correct_xgb_acc < log_acc:
    print("\nCon base en la precisión, el modelo de Regresión Logística es mejor.")
else:
    print("\nAmbos modelos tienen una precisión similar.")

print("\nEs importante destacar que el XGBoost muestra un mejor recall en la clase mayoritaria ('CHOQUE').")
print("Ambos modelos tienen dificultades con las clases minoritarias, lo que podría requerir un manejo adicional del desequilibrio de clases.")

Precisión (Accuracy) de Regresión Logística: 12.06%
Precisión (Accuracy) de XGBoost (corregida): 73.20%

Con base en la precisión, el modelo XGBoost es mejor.

Es importante destacar que el XGBoost muestra un mejor recall en la clase mayoritaria ('CHOQUE').
Ambos modelos tienen dificultades con las clases minoritarias, lo que podría requerir un manejo adicional del desequilibrio de clases.
